# Are TAD boudaries accessible in fiber-seq data?

TAD boundaries are often associated with CTCF binding sites, which can be identified in fiber-seq data. To determine if TAD boundaries are accessible in fiber-seq data, we can analyze TAD boundary calls from Rao et al. 2014 and compare them with CTCF binding sites identified in fiber-seq data

the TAD boundary calls are downloaded from https://ftp.ncbi.nlm.nih.gov/geo/series/GSE63nnn/GSE63525/suppl/GSE63525%5FGM12878%5Fprimary%2Breplicate%5FArrowhead%5Fdomainlist%2Etxt%2Egz

In [4]:
# load packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from scipy import stats
from statsmodels.stats.multitest import multipletests   

In [ ]:
# install pip install pyGenomeTracks if not already installed
# !pip install pyGenomeTracks


1. Inspect the TAD boundary calls
 - documentation for arrowhead output: https://github.com/aidenlab/juicer/wiki/Arrowhead

 the file contains the following columns:
 - chromosome1    x1    x2    chromosome2    y1    y2    color    corner_score    Uvar    Lvar    Usign    Lsign
 - Uvar = variance of the values within the upper half of the arrowhead matrix --> TADs are characterized by uniform, high-frequency internal contacts. If a region is a "true" TAD, the values within the triangle should be relatively consistent. High variance (Uvar) might indicate noisy data or a non-uniform structure that doesn't fit the classic TAD model
 - Lvar = variance of the values within the lower half of the arrowhead matrix
 - Usign = measures the "directionality" of the transformation. Arrowhead works on a gradient matrix (where values are transformed based on their neighbors). By multiplying the sum of signs by $-1$, a high positive Usign value indicates that the upper triangle correctly contains the expected negative gradient flow. This means that the contact frequencies decrease as you move away from the diagonal, which is a hallmark of TADs. A high positive Usign suggests that the identified domain has a strong directional signal consistent with TAD structure, while a low or negative Usign might indicate a weaker or less defined TAD boundary.
 - Lsign = similar to Usign but for the lower triangle. A high positive Lsign value indicates that the lower triangle correctly contains the expected positive gradient flow, which is also a hallmark of TADs. A high positive Lsign suggests that the identified domain has a strong directional signal consistent with TAD structure, while a low or negative Lsign might indicate a weaker or less defined T

In [12]:
# annotation directory
annotation_dir = "/project/spott/cshan/annotations"

domain_file = os.path.join(annotation_dir, "GSE63525_GM12878_primary+replicate_Arrowhead_domainlist.txt.gz")
domain_df = pd.read_csv(domain_file, sep="\t", header=None, names=["chr1", "x1", "x2", "chr2", "y1", "y2", "color", "corner_score", "Uvar", "Lvar", "Usign", "Lsign"])

# remove the first row for duplicate header
domain_df = domain_df[domain_df["chr1"] != "chr1"]
domain_df.shape




(9274, 12)

In [14]:
domain_df.head()

,chr1,x1,x2,chr2,y1,y2,color,corner_score,Uvar,Lvar,Usign,Lsign
1,1,144835000,145835000,1,144835000,145835000,"255,255,0",0.5517,0.35009,0.2686,0.4308,0.5391
2,1,68985000,70260000,1,68985000,70260000,"255,255,0",0.34374,0.27084,0.29571,0.40022,0.50935
3,1,49365000,50810000,1,49365000,50810000,"255,255,0",1.0567,0.24008,0.24148,0.49497,0.68434
4,1,163360000,164895000,1,163360000,164895000,"255,255,0",1.157,0.21904,0.24336,0.71738,0.60165
5,1,247800000,248395000,1,247800000,248395000,"255,255,0",0.33524,0.18309,0.27343,0.40819,0.40678


In [15]:
# both Usign and Lsign are positive for TADs, so we can filter for TADs with Usign > 0 and Lsign > 0

# change the data type of Usign and Lsign to numeric
domain_df["Usign"] = pd.to_numeric(domain_df["Usign"], errors="coerce")
domain_df["Lsign"] = pd.to_numeric(domain_df["Lsign"], errors="coerce") 

domain_df = domain_df[(domain_df["Usign"] > 0) & (domain_df["Lsign"] > 0)]
domain_df.shape

(9274, 12)

2. MSP = methyltransferase-sensitive patch, which is a region of DNA that is accessible to methyltransferase enzymes
- /project/spott/1_Shared_projects/LCL_Fiber_seq/FIRE/results/AL10_bc2178_19130/extracted_results/msp_by_chr/AL10_bc2178_19130.ft_extracted_msp.chr{1-22, X, Y}.bed.gz
- running this in the notebook crash the kernel, so I will run this in a separate script and save the output for pyGenomeTracks: ///project/spott/cshan/fiber-seq/code/TADs_Fiber_MSP_code/get_msp_block.py

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# read example accessible region from MSP file (BED12)
msp_file = "/project/spott/1_Shared_projects/LCL_Fiber_seq/FIRE/results/AL10_bc2178_19130/extracted_results/msp_by_chr/AL10_bc2178_19130.ft_extracted_msp.chr1.bed.gz"
msp_cols = [
    "chr", "start", "end", "fiber", "score", "strand",
    "thickStart", "thickEnd", "itemRgb", "blockCount", "blockSizes", "blockStarts"
]

msp_df = pd.read_csv(msp_file, sep='\t', header=None, names=msp_cols)


# expand BED12 blocks into one BED row per block for pyGenomeTracks
# str(v): ensure its a string
# .strip(','): remove leading/trailing commas
# .split(','): split by comma into list
# if x != '': filter out empty strings (which can occur if there are trailing commas)
def _to_int_list(v):
    vals = [x for x in str(v).strip(',').split(',') if x != '']
    # convert to integers
    return [int(x) for x in vals]

records = []

# loop row rows and expand blocks
for row in msp_df.itertuples(index=False):
    rel_starts = _to_int_list(row.blockStarts)
    sizes = _to_int_list(row.blockSizes)

    if len(rel_starts) != len(sizes):
        continue

    abs_starts = [row.start + s for s in rel_starts]
    abs_ends = [s + size for s, size in zip(abs_starts, sizes)]

    for i, (bs, be) in enumerate(zip(abs_starts, abs_ends), start=1):
        records.append({
            "chr": row.chr,
            "start": bs,
            "end": be,
            "name": f"{row.fiber}_block{i}",
            "score": row.score,
            "strand": row.strand
        })

msp_blocks_df = pd.DataFrame.from_records(records)
msp_blocks_df = msp_blocks_df.sort_values(["chr", "start", "end"]).reset_index(drop=True)

# write outputs for pyGenomeTracks
out_dir = "/project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pyGenomeTracks_inputs"
os.makedirs(out_dir, exist_ok=True)

# 1) expanded block-level BED6 (one row per block)
blocks_bed = os.path.join(out_dir, "msp_blocks_expanded.bed")
msp_blocks_df[["chr", "start", "end", "name", "score", "strand"]].to_csv(
    blocks_bed, sep='\t', header=False, index=False
)

# 2) cleaned original BED12
bed12_file = os.path.join(out_dir, "msp_fibers.bed12")
msp_df[msp_cols].to_csv(bed12_file, sep='\t', header=False, index=False)

print(f"Wrote block-level BED: {blocks_bed}")
print(f"Wrote BED12: {bed12_file}")
msp_blocks_df.head()


: 

In [7]:
!sbatch /project/spott/cshan/fiber-seq/code/TADs_Fiber_MSP_code/get_msp_block.sh

sbatch: Verify job submission ...
sbatch: Using a shared partition ...
sbatch: Partition: caslake
sbatch: QOS-Flag: caslake
sbatch: Account: pi-spott
sbatch: Verification: ***PASSED***
Submitted batch job 48804796


In [ ]:
import pysam
import os

# ── Source MSP files (one per chromosome) ─────────────────────────────────
msp_dir    = ("/project/spott/1_Shared_projects/LCL_Fiber_seq/FIRE/results/"
              "AL10_bc2178_19130/extracted_results/msp_by_chr")
sample     = "AL10_bc2178_19130"
chromosomes = [f"chr{i}" for i in list(range(1, 23)) + ["X", "Y"]]

# ── Output ────────────────────────────────────────────────────────────────
out_dir = "/project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pyGenomeTracks_inputs"
os.makedirs(out_dir, exist_ok=True)
out_bed12 = os.path.join(out_dir, "msp_fibers.bed12")

# ── Stream each chr file and write rows directly to output BED12 ─────────────────────────
# pysam.TabixFile fetches records via the .tbi index without loading the

total = 0
with open(out_bed12, "w") as out_fh:
    for chrom in chromosomes:
        src = os.path.join(msp_dir, f"{sample}.ft_extracted_msp.{chrom}.bed.gz")
        if not os.path.exists(src):
            continue
        with pysam.TabixFile(src) as tbx:
            for rec in tbx.fetch(chrom):
                out_fh.write(rec + "\n")
                total += 1

print(f"Wrote {total:,} fiber rows to {out_bed12}")


Wrote 5,664,746 fiber rows to /project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pyGenomeTracks_inputs/msp_fibers.bed12


In [32]:
# inspect bed file outputs

# select a few rows to check the format instead of reading the entire file into memory 

out_dir = "/project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pyGenomeTracks_inputs"
msp_blocks_bed = os.path.join(out_dir, "msp_fibers.bed12")

!head -n 5 "$msp_blocks_bed"

chr1	10001	28189	m84241_250311_201728_s2/164237536/ccs	4	-	10001	28189	147,112,219	92	0,55,18,47,47,22,39,48,42,31,60,105,69,77,91,76,104,83,84,1,19,42,8,57,28,7,48,34,46,42,107,69,43,37,7,40,93,105,67,66,46,71,71,105,86,68,55,81,36,39,109,38,70,46,56,42,44,60,33,50,59,47,92,10,24,108,72,96,70,26,79,59,3,14,46,34,1,1,30,128,35,31,13,62,58,35,8,61,129,194,16,1	0,31,252,431,580,953,1096,1289,1481,1660,1881,2080,2282,2440,2645,2882,3066,3251,3461,3680,3845,4008,4217,4374,4574,4758,4915,5085,5265,5476,5638,5850,6051,6274,6471,6648,6826,7033,7284,7478,7697,7889,8104,8321,8564,8798,9011,9214,9437,9641,9777,10031,10218,10444,10636,10846,11029,11190,11407,11599,11779,11988,12160,12382,12577,12698,12919,13123,13485,13725,13920,14134,14351,14538,14705,14893,15043,15155,15282,15468,15828,16026,16199,16371,16561,16787,16972,17140,17276,17564,17933,18187
chr1	10010	25652	m84241_241120_172940_s3/79760148/ccs	6	+	10010	25652	147,112,219	84	0,73,36,14,1,95,10,241,7,15,1,53,16,9,70,102,59,58,75,39,32,3

I tried plotting at Mb sizes, but no fibers are able to span the whole region, so I want to plot smaller TADs

In [6]:
import pandas as pd

annotation_dir = "/project/spott/cshan/annotations"
domain_file    = f"{annotation_dir}/GSE63525_GM12878_primary+replicate_Arrowhead_domainlist.txt.gz"

tad_df = pd.read_csv(domain_file, sep="\t",
                     names=["chr","x1","x2","chr2","y1","y2","color",
                             "corner_score","Uvar","Lvar","Usign","Lsign"],
                     header=0)

# keep diagonal domains only, fix chr prefix, filter quality
tad_df = tad_df[tad_df["chr"] == tad_df["chr2"]].copy()
tad_df["chr"]   = tad_df["chr"].astype(str).apply(lambda v: v if v.startswith("chr") else f"chr{v}")
tad_df["start"] = pd.to_numeric(tad_df["x1"], errors="coerce").astype("Int64")
tad_df["end"]   = pd.to_numeric(tad_df["x2"], errors="coerce").astype("Int64")
tad_df["Usign"] = pd.to_numeric(tad_df["Usign"], errors="coerce")
tad_df["Lsign"] = pd.to_numeric(tad_df["Lsign"], errors="coerce")
tad_df = tad_df.dropna(subset=["start","end"])
tad_df = tad_df[(tad_df["Usign"] > 0) & (tad_df["Lsign"] > 0) & (tad_df["end"] > tad_df["start"])]

# rank by size: smallest TADs first (plotted at top of read panel)
tad_df["size"] = tad_df["end"] - tad_df["start"]
tad_df = tad_df.sort_values("size").reset_index(drop=True)

print(f"TADs loaded: {len(tad_df):,}")
tad_df[["chr","start","end","size","corner_score"]].head(10)


TADs loaded: 9,274


,chr,start,end,size,corner_score
0,chrX,64885000,64950000,65000,0.85835
1,chr22,36785000,36850000,65000,0.70554
2,chr6,34150000,34215000,65000,0.81731
3,chr15,74840000,74905000,65000,0.72369
4,chr21,46495000,46560000,65000,0.75466
5,chr3,112175000,112240000,65000,0.75343
6,chr10,64965000,65030000,65000,0.67707
7,chr10,71005000,71070000,65000,0.71608
8,chr9,126895000,126960000,65000,0.76064
9,chr1,2520000,2585000,65000,0.73729


In [3]:
import pandas as pd

annotation_dir = "/project/spott/cshan/annotations"
domain_file    = f"{annotation_dir}/GSE63525_GM12878_primary+replicate_Arrowhead_domainlist.txt.gz"

tad_df = pd.read_csv(domain_file, sep="\t",
                     names=["chr","x1","x2","chr2","y1","y2","color",
                             "corner_score","Uvar","Lvar","Usign","Lsign"],
                     header=0)

# filter for chr1 only
tad_chr1_df = tad_df[tad_df["chr"] == "1"].copy()
tad_chr1_df.shape
tad_chr1_df.head()


,chr,x1,x2,chr2,y1,y2,color,corner_score,Uvar,Lvar,Usign,Lsign
0,1,144835000,145835000,1,144835000,145835000,"255,255,0",0.55170,0.35009,0.26860,0.43080,0.53910
1,1,68985000,70260000,1,68985000,70260000,"255,255,0",0.34374,0.27084,0.29571,0.40022,0.50935
2,1,49365000,50810000,1,49365000,50810000,"255,255,0",1.05670,0.24008,0.24148,0.49497,0.68434
3,1,163360000,164895000,1,163360000,164895000,"255,255,0",1.15700,0.21904,0.24336,0.71738,0.60165
4,1,247800000,248395000,1,247800000,248395000,"255,255,0",0.33524,0.18309,0.27343,0.40819,0.40678


3. Export a TAD-selected region for plotting TAD track on top + fiber/MSP track
- The biggest problem is that the TADs are too big to be spanned by fibers, and no single fiber can span the whole TAD
- Instead of plotting the whole TAD, I want to plot TAD boundaries defined as +/- 10bp of the TAD boundary calls and reduce the window size


In [5]:
# all chr, all tads (no regional filter)
tad_domains_region = tad_df.copy().reset_index(drop=True)

boundary_width = 10
tad_start_boundaries = tad_domains_region.assign(
    b_start=tad_domains_region["start"],
    b_end=tad_domains_region["start"] + boundary_width,
    boundary="start",
)
tad_end_boundaries = tad_domains_region.assign(
    b_start=tad_domains_region["end"],
    b_end=tad_domains_region["end"] + boundary_width,
    boundary="end",
)

tad_region = pd.concat(
    [
        tad_start_boundaries[["chr", "b_start", "b_end", "boundary"]],
        tad_end_boundaries[["chr", "b_start", "b_end", "boundary"]],
    ],
    ignore_index=True
).rename(columns={"b_start": "start", "b_end": "end"}).sort_values(["chr", "start", "end"]).reset_index(drop=True)

print(f"TADs all-chr: {len(tad_domains_region):,} domains, {len(tad_region):,} boundaries")

# sav TAD region bed
out_dir = "/project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/TAD_fiber_tracks_inputs"
tad_region_bed = os.path.join(out_dir, "tad_boundaries.bed")
tad_region[["chr", "start", "end", "boundary"]].to_csv(
    tad_region_bed, sep='\t', header=False, index=False
)

TADs all-chr: 9,274 domains, 18,548 boundaries


In [42]:
# examine tad boundary bed

import pandas as pd

input_bed = "/project/spott/cshan/annotations/hic/ENCFF156ECM.bedpe.gz"
tad_region = pd.read_csv(input_bed, sep='\t', header=None, names=["chr", "start", "end", "boundary"])
tad_region.shape
tad_region.iloc[10:20]

,chr,start,end,boundary
10,chr1,1985000,1985010,end
11,chr1,2055000,2055010,end
12,chr1,2120000,2120010,start
13,chr1,2320000,2320010,end
14,chr1,2345000,2345010,start
15,chr1,2475000,2475010,end
16,chr1,2520000,2520010,start
17,chr1,2585000,2585010,end
18,chr1,2710000,2710010,start
19,chr1,3340000,3340010,end


In [1]:
%%bash

cat > /project/spott/cshan/fiber-seq/code/TADs_Fiber_MSP_code/convert_hic_formats.sh << 'EOF'
#!/bin/bash
#SBATCH --job-name=convert_hic_formats
#SBATCH --time=12:00:00
#SBATCH --mem=120G
#SBATCH --cpus-per-task=4
#SBATCH --account=pi-spott
#SBATCH --partition=caslake
#SBATCH --output=/project/spott/cshan/fiber-seq/results/logs/convert_hic_formats_%j.out
#SBATCH --error=/project/spott/cshan/fiber-seq/results/logs/convert_hic_formats_%j.err

module load python/miniforge-25.3.0
source activate /project/spott/cshan/envs/hicexplorer

# hic to cool
hicConvertFormat \
  -m /project/spott/cshan/annotations/GSE63525_GM12878_insitu_primary+replicate_combined_30.hic \
  --inputFormat hic \
  --outputFormat cool \
  -o /project/spott/cshan/annotations/hic/ENCFF216QQM.cool

EOF

chmod +x /project/spott/cshan/fiber-seq/code/TADs_Fiber_MSP_code/convert_hic_formats.sh


In [13]:
!sbatch /project/spott/cshan/fiber-seq/code/TADs_Fiber_MSP_code/convert_hic_formats.sh


sbatch: Verify job submission ...
sbatch: Using a shared partition ...
sbatch: Partition: caslake
sbatch: QOS-Flag: caslake
sbatch: Account: pi-spott
sbatch: Verification: ***PASSED***
Submitted batch job 48809590




# Overlay with CTCF binding sites, DNAse accessibility
- CTCF: /project/spott/cshan/annotations/CTCF_ENCFF636ODI.bigWig
- DNAse: /project/spott/cshan/annotations/DNase.ENCFF743ULW.bigWig
- CAGE: /project/spott/cshan/annotations/fantom5/fantom5.hg38.LCL.consensus.CAGE_peaks.bed.gz
- MSP: /project/spott/1_Shared_projects/LCL_Fiber_seq/FIRE/results/AL10_bc2178_19130/extracted_results/msp_by_chr/AL10_bc2178_19130.ft_extracted_msp.chr1.bed.gz
- TAD boundaries: /project/spott/cshan/annotations/hic/ENCFF156ECM.bedpe.gz

For now, all the analysis are done on AL10_bc2178_19130 chr1

The smallest TAD is 6kb, but individual fiber reads is 10-20kb, not a single read can span the whole TAD window

## pyGenomeTracks Overlay: TAD Boundaries + MSP + CTCF + DNase + CAGE (chr1)
Use this section to visualize narrow TAD boundary calls with signal/context tracks.
By default, it picks one boundary from `tad_boundaries.bed` on `chr1` and plots a wider flanking window.


In [8]:
from pathlib import Path
import pandas as pd
import subprocess
import shutil
import pysam

OUT_DIR = Path('/project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pygenometracks_overlay/inputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Convert Arrowhead TAD domainlist to BED6 ────────────────────────────
domain_file = Path('/project/spott/cshan/annotations/HiC_GSE63525_LCL/GSE63525_GM12878_primary+replicate_Arrowhead_domainlist.txt.gz')

tad_df = pd.read_csv(domain_file, sep='\t',
                     names=['chr','x1','x2','chr2','y1','y2','color',
                             'corner_score','Uvar','Lvar','Usign','Lsign'],
                     header=0)
tad_df = tad_df[tad_df['chr'] == tad_df['chr2']].copy()
tad_df['chr']   = tad_df['chr'].astype(str).apply(lambda v: v if v.startswith('chr') else f'chr{v}')
tad_df['start'] = pd.to_numeric(tad_df['x1'], errors='coerce').astype('Int64')
tad_df['end']   = pd.to_numeric(tad_df['x2'], errors='coerce').astype('Int64')
tad_df['Usign'] = pd.to_numeric(tad_df['Usign'], errors='coerce')
tad_df['Lsign'] = pd.to_numeric(tad_df['Lsign'], errors='coerce')
tad_df = tad_df.dropna(subset=['start','end'])
tad_df = tad_df[(tad_df['Usign'] > 0) & (tad_df['Lsign'] > 0) & (tad_df['end'] > tad_df['start'])]
tad_df['name']  = 'TAD_' + pd.RangeIndex(len(tad_df)).astype(str)
tad_df['score'] = 0  # BED score field; not used for coloring here
tad_df['strand'] = '.'

tad_bed = OUT_DIR / 'tads.bed'
tad_df[['chr','start','end','name','score','strand']].sort_values(['chr','start']).to_csv(
    tad_bed, sep='\t', header=False, index=False
)
print(f'Wrote {len(tad_df):,} TADs → {tad_bed}')

# ── 2. Extend 1-bp CAGE peaks to 200 bp (±100 bp around peak midpoint) ────
# Reason: FANTOM5 CAGE peaks are called at single-nucleotide resolution (1 bp wide).
# pyGenomeTracks renders them as rectangles; at any window >~100 bp they are
# invisible. Extending ±100 bp makes them visible without distorting their positions.
CAGE_IN  = Path('/project/spott/cshan/annotations/fantom5/fantom5.hg38.LCL.consensus.CAGE_peaks.bed.gz')
cage_ext = OUT_DIR / 'cage_lcl_extended.bed.gz'

cage_df = pd.read_csv(CAGE_IN, sep='\t', header=None,
                      names=['chr','start','end','name','score','strand',
                             'thickStart','thickEnd','rgb'],
                      usecols=range(6))
cage_df.columns = ['chr','start','end','name','score','strand']

HALF = 100
cage_df['mid']   = (cage_df['start'] + cage_df['end']) // 2
cage_df['start'] = (cage_df['mid'] - HALF).clip(lower=0)
cage_df['end']   = cage_df['mid'] + HALF
cage_df = cage_df.drop(columns='mid').sort_values(['chr','start'])

cage_tmp = OUT_DIR / 'cage_lcl_extended.bed'
cage_df[['chr','start','end','name','score','strand']].to_csv(
    cage_tmp, sep='\t', header=False, index=False
)
bgzip_path = shutil.which('bgzip')
if bgzip_path is not None:
    subprocess.run([bgzip_path, '-f', str(cage_tmp)], check=True)
else:
    # fallback when bgzip binary is unavailable
    pysam.tabix_compress(str(cage_tmp), str(cage_ext), force=True)
tabix_path = shutil.which('tabix')
if tabix_path is not None:
    subprocess.run([tabix_path, '-f', '-p', 'bed', str(cage_ext)], check=True)
else:
    # fallback when tabix binary is unavailable
    pysam.tabix_index(str(cage_ext), preset='bed', force=True)
print(f'Wrote extended CAGE peaks → {cage_ext}')


Wrote 9,274 TADs → /project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pygenometracks_overlay/inputs/tads.bed
Wrote extended CAGE peaks → /project/spott/cshan/fiber-seq/results/TADs_m6a_footprint_overlay/inputs/cage_lcl_extended.bed.gz


Instead of randomly picking region, I want to plot promoter regions

In [9]:
# open GENCODE defined TSS

import os
import pandas as pd

annotation_dir = "/project/spott/cshan/annotations"
gencode_file = os.path.join(annotation_dir, "TSS.gencode.v49.bed")
gencode_TSS_df = pd.read_csv(gencode_file, sep='\t', header=None, names=['chr','start','end','name','score','strand'])

gencode_TSS_df.head()

,chr,start,end,name,score,strand
0,chr1,11120,11121,ENSG00000290825.2,.,+
1,chr1,12009,12010,ENSG00000223972.6,.,+
2,chr1,30743,30744,ENSG00000310526.1,.,-
3,chr1,24885,24886,ENSG00000227232.6,.,-
4,chr1,17435,17436,ENSG00000278267.1,.,-


In [6]:
from pathlib import Path
import pandas as pd

OUT_DIR   = Path('/project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pygenometracks_overlay')
INPUT_DIR = OUT_DIR / 'inputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Region ----------------------------------------------------------------
# How the window size is determined:
#   REGION_OVERRIDE is a free-form genomic interval string 'chr:start-end'.
#   The window size = end - start.  Guidelines:
#     - For a single TAD boundary context:  ~20–50 kb   (e.g. ±25 kb around center)
#     - For one full TAD:                   TAD size itself, typically 100 kb – 1 Mb
#     - For multi-TAD overview:             1–3 Mb
#   The Hi-C depth parameter (500 000 bp) caps how far off-diagonal the matrix shows.
#   For sub-TAD windows (<100 kb) consider lowering MCOOL_RES to 5000 (5 kb).
REGION_OVERRIDE = 'chr1:100000-150000'   

# ---- Inputs ----------------------------------------------------------------
CTCF_BW  = Path('/project/spott/cshan/annotations/CTCF_ENCFF636ODI.bigWig')
DNASE_BW = Path('/project/spott/cshan/annotations/DNase.ENCFF743ULW.bigWig')
CAGE_BED = INPUT_DIR / 'cage_lcl_extended.bed.gz'
M6A_BED  = Path('/project/spott/1_Shared_projects/LCL_Fiber_seq/FIRE/results/AL10_bc2178_19130/extracted_results/m6a_by_chr/AL10_bc2178_19130.ft_extracted_m6a.chr1.bed.gz')
CPG_BW   = Path('/project/spott/cshan/fiber-seq/results/fire_CpG/AL10_bc2178_19130/'
                'AL10_bc2178_19130_CPG.combined.bw')
TAD_BED  = INPUT_DIR / 'tads.bed'
MCOOL    = Path('/project/spott/cshan/annotations/hic/ENCFF216QQM.cool')
MCOOL_RES = 5000   

FIRE_DIR = Path('/project/spott/cshan/fiber-seq/results/PolII/FIRE_combined_footprints/'
                'joint_trained_tracks/chr1')

# Small footprints (~10-30bp) = TF binding; large (~140-160bp) = nucleosomes.
FIRE_TRACKS = [
    ('10-30bp',  FIRE_DIR / 'combined_chr1_10-30bp_fp_cov.bw',  '#08306b'),
    ('40-60bp',  FIRE_DIR / 'combined_chr1_40-60bp_fp_cov.bw',  '#c6dbef'),
    ('60-80bp',  FIRE_DIR / 'combined_chr1_60-80bp_fp_cov.bw',  '#6baed6'),
    ('140-160bp',FIRE_DIR / 'combined_chr1_140-160bp_fp_cov.bw','#4292c6'),
]

# ---- Scaling ---------------------------------------------------------------
BW_MIN = 0
BW_MAX = 'auto'   # each bigWig scales to its own local max


# ---- Outputs ---------------------------------------------------------------
tracks_ini = OUT_DIR / 'chr1.overlay.tracks.ini'
region_tag = REGION_OVERRIDE.replace(':', '_').replace('-', '_')
out_png    = OUT_DIR / f'{region_tag}.overlay.png'
out_pdf    = OUT_DIR / f'{region_tag}.overlay.pdf'

# ---- Build tracks.ini ------------------------------------------------------
lines = []

def section(name, props):
    lines.append(f'[{name}]')
    for k, v in props.items():
        lines.append(f'{k} = {v}')
    lines.append('')

def spacer(height=0.3):
    lines.append('[spacer]')
    lines.append(f'height = {height}')
    lines.append('')

def bw_section(name, title, bw_path, color, height=2):
    props = {
        'file':            str(bw_path),
        'title':           title,
        'height':          str(height),
        'color':           color,
        'min_value':       str(BW_MIN),
        'show_data_range': 'true',
        'file_type':       'bigwig',
    }
    if BW_MAX != 'auto':
        props['max_value'] = str(BW_MAX)
    section(name, props)

# x-axis at top
section('x-axis', {'fontsize': '10', 'where': 'top'})

# Hi-C contact matrix (mcool)
section('hic', {
    'file':      f'{MCOOL}::/resolutions/{MCOOL_RES}',
    'title':     f'GM12878 Hi-C ({MCOOL_RES // 1000} kb)',
    'depth':     '5000',
    'height':    '6',
    'colormap':  'Reds',
    'file_type': 'hic_matrix',
})
spacer()

# TADs as domain rectangles (outline only, no fill)
section('tads', {
    'file':         str(TAD_BED),
    'title':        'TADs (Arrowhead, GM12878)',
    'height':       '2',
    'color':        'none',
    'border_color': '#e41a1c',
    'line_width':   '1.5',
    'labels':       'false',
    'display':      'interleaved',
    'file_type':    'bed',
})
spacer()

# CTCF
bw_section('ctcf', 'CTCF', CTCF_BW, 'black')
spacer()

# DNase
bw_section('dnase', 'DNase', DNASE_BW, '#1f77b4')
spacer()

# CAGE (peaks extended ±100 bp so they render at genomic scale)
section('cage', {
    'file':         str(CAGE_BED),
    'title':        'FANTOM5 CAGE',
    'height':       '1.5',
    'color':        '#2ca02c',
    'border_color': '#1b6e1b',
    'labels':       'false',
    'display':      'collapsed',
    'file_type':    'bed',
})
spacer()

# CpG methylation (bigWig)
bw_section('cpg', 'CpG methylation', CPG_BW, '#8856a7', height=1.5)
spacer()

# FiberHMM footprint bigWigs
for label, bw_path, color in FIRE_TRACKS:
    tid = 'fire_' + label.replace('-', '_')
    bw_section(tid, f'Footprint {label}', bw_path, color, height=1.2)
spacer()

# M6A 
section('m6a', {
    'file':         str(M6A_BED),
    'title':        'M6A ',
    'height':       '20',
    'color':        "#970aa9",
    'border_color': "#DAB7D4",
    'color_utr':    "#410642",
    'labels':       'false',
    'display':      'stacked',
    'file_type':    'bed',
})

tracks_ini.write_text('\n'.join(lines))

print(f'Wrote tracks config: {tracks_ini}')
print(f'Region:              {REGION_OVERRIDE}')
print(f'Output PNG:          {out_png}')
print(f'Output PDF:          {out_pdf}')


Wrote tracks config: /project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pygenometracks_overlay/chr1.overlay.tracks.ini
Region:              chr1:100000-150000
Output PNG:          /project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pygenometracks_overlay/chr1_100000_150000.overlay.png
Output PDF:          /project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pygenometracks_overlay/chr1_100000_150000.overlay.pdf


In [ ]:
%%bash

cat > /project/spott/cshan/fiber-seq/code/TADs_Fiber_MSP_code/plot_pygenometracks_overlay.sh << 'EOF'
#!/bin/bash
#SBATCH --job-name=plot_pgt_overlay
#SBATCH --time=02:00:00
#SBATCH --mem=48G
#SBATCH --cpus-per-task=2
#SBATCH --account=pi-spott
#SBATCH --partition=caslake
#SBATCH --output=/project/spott/cshan/fiber-seq/results/logs/plot_pgt_overlay_%j.out
#SBATCH --error=/project/spott/cshan/fiber-seq/results/logs/plot_pgt_overlay_%j.err

module load python/miniforge-25.3.0
source activate /project/spott/cshan/envs/Jupyter-notebook

TRACKS_INI=${TRACKS_INI:-/project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pygenometracks_overlay/chr1.overlay.tracks.ini}
REGION=${REGION:-chr1:100000-150000}
OUT_PNG=${OUT_PNG:-/project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pygenometracks_overlay/chr1_2707500_2757500.overlay.png}
OUT_PDF=${OUT_PDF:-/project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pygenometracks_overlay/chr1_2707500_2757500.overlay.pdf}

echo "Tracks: $TRACKS_INI"
echo "Region: $REGION"
echo "Output PNG: $OUT_PNG"
echo "Output PDF: $OUT_PDF"

pyGenomeTracks \
    --tracks "$TRACKS_INI" \
    --region "$REGION" \
    --outFileName "$OUT_PNG" \
    --dpi 220

pyGenomeTracks \
    --tracks "$TRACKS_INI" \
    --region "$REGION" \
    --outFileName "$OUT_PDF"

echo "Done. Rendered:"
echo "  $OUT_PNG"
echo "  $OUT_PDF"
EOF

chmod +x /project/spott/cshan/fiber-seq/code/TADs_m6a_footprint_overlay_code/archive/plot_pygenometracks_overlay.sh
echo "Wrote plot_pygenometracks_overlay.sh"


Wrote plot_pygenometracks_overlay.sh


In [ ]:
!sbatch /project/spott/cshan/fiber-seq/code/TADs_m6a_footprint_overlay_code/archive/plot_pygenometracks_overlay.sh

sbatch: Verify job submission ...
sbatch: Using a shared partition ...
sbatch: Partition: caslake
sbatch: QOS-Flag: caslake
sbatch: Account: pi-spott
sbatch: Verification: ***PASSED***
Submitted batch job 48834306


The above chunk serves as an example for plotting specific region on chr1 for AL10, I now want to change the code to allow user to define samples that are in `/project/spott/1_Shared_projects/LCL_Fiber_seq/Data/LCL_sample_metatable_merged_samples_31samples.csv` and define any `chr`

/project/spott/cshan/fiber-seq/code/TADs_m6a_footprint_overlay_code/


In [ ]:
%%R

# although the script only says m6a, it now includes 5mc

source("/project/spott/cshan/fiber-seq/code/footprintR_modbam_code/modbam_footprintR_functions.R")

parse_cli_args <- function(argv) {
  out <- list(
    sample = NULL,
    chrom = NULL,
    start = NULL,
    end = NULL,
    region = NULL,
    metadata = "/project/spott/1_Shared_projects/LCL_Fiber_seq/Data/LCL_sample_metatable_merged_samples_31samples.csv",
    output_root = "/project/spott/cshan/fiber-seq/results/TADs_m6a_footprint_overlay",
    chrom_sizes = "/project/spott/cshan/annotations/hg38.chrom.sizes",
    mod_prob_threshold = 0.9
  )

  i <- 1L
  while (i <= length(argv)) {
    arg <- argv[[i]]
    if (grepl("^--[^=]+=", arg)) {
      pieces <- strsplit(sub("^--", "", arg), "=", fixed = TRUE)[[1]]
      key <- pieces[[1]]
      value <- paste(pieces[-1], collapse = "=")
    } else if (grepl("^--", arg)) {
      key <- sub("^--", "", arg)
      if (i == length(argv)) {
        stop(sprintf("Missing value for argument: %s", arg), call. = FALSE)
      }
      i <- i + 1L
      value <- argv[[i]]
    } else if (grepl("=", arg, fixed = TRUE)) {
      pieces <- strsplit(arg, "=", fixed = TRUE)[[1]]
      key <- pieces[[1]]
      value <- paste(pieces[-1], collapse = "=")
    } else {
      stop(sprintf("Unrecognized argument: %s", arg), call. = FALSE)
    }

    out[[key]] <- value
    i <- i + 1L
  }

  out
}

parse_region_string <- function(region) {
  m <- regexec("^([^:]+):(\\d+)-(\\d+)$", region)
  parts <- regmatches(region, m)[[1]]
  if (length(parts) != 4L) {
    stop("Region must look like chr1:10000-20000", call. = FALSE)
  }

  list(
    chrom = parts[[2]],
    start = as.integer(parts[[3]]),
    end = as.integer(parts[[4]])
  )
}

normalize_chrom <- function(chrom) {
  chrom <- as.character(chrom)
  if (startsWith(chrom, "chr")) chrom else paste0("chr", chrom)
}

args <- parse_cli_args(commandArgs(trailingOnly = TRUE))

if (!is.null(args$region) && (is.null(args$chrom) || is.null(args$start) || is.null(args$end))) {
  region_parts <- parse_region_string(args$region)
  args$chrom <- region_parts$chrom
  args$start <- region_parts$start
  args$end <- region_parts$end
}

if (is.null(args$sample) || is.null(args$chrom) || is.null(args$start) || is.null(args$end)) {
  stop("Provide --sample plus either --region or --chrom/--start/--end.", call. = FALSE)
}

sample_name       <- as.character(args$sample)
chrom             <- normalize_chrom(args$chrom)
region_start      <- as.integer(args$start)
region_end        <- as.integer(args$end)
mod_prob_threshold <- as.numeric(args$mod_prob_threshold)
chrom_sizes_path  <- as.character(args$chrom_sizes)

if (is.na(region_start) || is.na(region_end) || region_start < 1L || region_end < region_start) {
  stop("Invalid region coordinates.", call. = FALSE)
}

if (!file.exists(chrom_sizes_path)) {
  stop(sprintf("chrom sizes file not found: %s", chrom_sizes_path), call. = FALSE)
}

metadata_path <- args$metadata
output_root   <- args$output_root
region_slug   <- sprintf("%s_%d_%d", chrom, region_start, region_end)
region_dir    <- file.path(output_root, sample_name, chrom, region_slug)
dir.create(region_dir, recursive = TRUE, showWarnings = FALSE)

meta <- data.table::fread(metadata_path)
sample_row <- meta[meta$sample_name == sample_name, ]
if (!nrow(sample_row)) {
  stop(sprintf("Sample %s was not found in %s", sample_name, metadata_path), call. = FALSE)
}
sample_row <- sample_row[1]

bam_path <- sample_row$bam_file[[1]]
assert_file_exists(bam_path, "sample BAM")
assert_file_exists(paste0(bam_path, ".bai"), "sample BAM index")

bam_seqinfo <- read_bam_seqinfo(bam_path)
if (!(chrom %in% GenomeInfoDb::seqnames(bam_seqinfo))) {
  stop(sprintf("Chromosome %s is absent from %s", chrom, bam_path), call. = FALSE)
}

region_gr <- GenomicRanges::GRanges(
  seqnames = chrom,
  ranges = IRanges::IRanges(start = region_start, end = region_end)
)
GenomeInfoDb::seqinfo(region_gr) <- bam_seqinfo[chrom]
region_gr <- GenomicRanges::trim(region_gr)

region_info <- data.table::data.table(
  sample_name = sample_name,
  chromosome  = chrom,
  region_start = start(region_gr),
  region_end   = end(region_gr),
  bam_path     = bam_path,
  metadata_path = metadata_path
)
write_gz_tsv(region_info, file.path(region_dir, sprintf("%s.%s.region_info.tsv.gz", sample_name, region_slug)))

position_dt <- read_modbam_summary_positions(
  bamfiles          = setNames(bam_path, sample_name),
  sample_name       = sample_name,
  promoter_windows  = region_gr,
  modbase_code      = "a",
  seqinfo           = bam_seqinfo,
  bpparam           = BiocParallel::SerialParam(),
  modProbThreshold  = mod_prob_threshold
)

position_tsv <- file.path(region_dir, sprintf("%s.%s.m6A_position_calls.tsv.gz", sample_name, region_slug))
write_gz_tsv(position_dt, position_tsv)

bedgraph_path <- file.path(region_dir, sprintf("%s.%s.m6A_fraction.bedgraph", sample_name, region_slug))
if (nrow(position_dt)) {
  bedgraph_dt <- position_dt[, .(
    score = mean(frac_modified, na.rm = TRUE)
  ), by = .(chrom = call_chromosome, start0 = position - 1L, end = position)]
  bedgraph_dt <- bedgraph_dt[order(chrom, start0, end)]
} else {
  bedgraph_dt <- data.table::data.table(
    chrom  = character(),
    start0 = integer(),
    end    = integer(),
    score  = numeric()
  )
}
data.table::fwrite(bedgraph_dt, file = bedgraph_path, sep = "\t", col.names = FALSE)

# Convert bedGraph to bigWig using the genome-wide hg38.chrom.sizes.
# The old approach wrote a per-run chrom sizes file derived from the BAM
# seqinfo, which often had missing or incorrect chromosome lengths, causing
# bedGraphToBigWig to exit with status 255.
bigwig_path <- file.path(region_dir, sprintf("%s.%s.m6A_fraction.bw", sample_name, region_slug))

bedgraph_to_bigwig <- Sys.which("bedGraphToBigWig")
if (!nzchar(bedgraph_to_bigwig) && file.exists("/project/spott/cshan/tools/bedGraphToBigWig")) {
  bedgraph_to_bigwig <- "/project/spott/cshan/tools/bedGraphToBigWig"
}

if (nzchar(bedgraph_to_bigwig) && nrow(bedgraph_dt)) {
  conv_result <- system2(
    command = bedgraph_to_bigwig,
    args    = c(bedgraph_path, chrom_sizes_path, bigwig_path),
    stdout  = TRUE,
    stderr  = TRUE
  )
  conv_status <- attr(conv_result, "status")
  if (!is.null(conv_status) && conv_status != 0L) {
    warning(sprintf(
      "bedGraphToBigWig exited with status %d for %s. bigWig will not be available.\nOutput:\n%s",
      conv_status, bigwig_path, paste(conv_result, collapse = "\n")
    ))
    # Remove the broken output file so downstream code does not try to open it
    if (file.exists(bigwig_path)) file.remove(bigwig_path)
    bigwig_path <- NULL
  }
} else {
  bigwig_path <- NULL
}

plot_path <- file.path(region_dir, sprintf("%s.%s.m6A_fraction.pdf", sample_name, region_slug))
if (nrow(position_dt)) {
  p <- ggplot2::ggplot(position_dt, ggplot2::aes(x = position, y = frac_modified)) +
    ggplot2::geom_line(color = "#7B3294", linewidth = 0.5) +
    ggplot2::coord_cartesian(xlim = c(start(region_gr), end(region_gr)), ylim = c(0, 1)) +
    ggplot2::labs(
      title = sprintf("Region m6A fraction: %s %s", sample_name, region_slug),
      x     = sprintf("Genomic position on %s", chrom),
      y     = "Fraction modified"
    ) +
    ggplot2::theme_bw(base_size = 11)
} else {
  p <- ggplot2::ggplot() +
    ggplot2::annotate("text", x = 0, y = 0, label = "No m6A positions with coverage in this region.") +
    ggplot2::theme_void() +
    ggplot2::labs(title = sprintf("Region m6A fraction: %s %s", sample_name, region_slug))
}
ggplot2::ggsave(plot_path, p, width = 8, height = 2.8)

summary_path <- file.path(region_dir, sprintf("%s.%s.m6A_region_summary.tsv.gz", sample_name, region_slug))
summary_dt <- if (nrow(position_dt)) {
  position_dt[, .(
    modified_calls          = sum(modified_calls, na.rm = TRUE),
    total_calls             = sum(total_calls, na.rm = TRUE),
    mean_fraction_modified  = mean(frac_modified, na.rm = TRUE),
    median_fraction_modified = stats::median(frac_modified, na.rm = TRUE),
    covered_positions       = .N
  )]
} else {
  data.table::data.table(
    modified_calls          = 0L,
    total_calls             = 0L,
    mean_fraction_modified  = NA_real_,
    median_fraction_modified = NA_real_,
    covered_positions       = 0L
  )
}
summary_dt[, `:=`(
  sample_name        = sample_name,
  chromosome         = chrom,
  region_start       = start(region_gr),
  region_end         = end(region_gr),
  mod_prob_threshold = mod_prob_threshold
)]
data.table::setcolorder(
  summary_dt,
  c(
    "sample_name", "chromosome", "region_start", "region_end", "mod_prob_threshold",
    "modified_calls", "total_calls", "mean_fraction_modified", "median_fraction_modified",
    "covered_positions"
  )
)
write_gz_tsv(summary_dt, summary_path)

message("Region m6A outputs written to: ", region_dir)

# ── 5mC (methylated-C) section ────────────────────────────────────────────────
position_dt_5mc <- read_modbam_summary_positions(
  bamfiles          = setNames(bam_path, sample_name),
  sample_name       = sample_name,
  promoter_windows  = region_gr,
  modbase_code      = "m",
  seqinfo           = bam_seqinfo,
  bpparam           = BiocParallel::SerialParam(),
  modProbThreshold  = mod_prob_threshold
)

bedgraph_path_5mc <- file.path(region_dir, sprintf("%s.%s.5mC_fraction.bedgraph", sample_name, region_slug))
if (nrow(position_dt_5mc)) {
  bedgraph_dt_5mc <- position_dt_5mc[, .(
    score = mean(frac_modified, na.rm = TRUE)
  ), by = .(chrom = call_chromosome, start0 = position - 1L, end = position)]
  bedgraph_dt_5mc <- bedgraph_dt_5mc[order(chrom, start0, end)]
} else {
  bedgraph_dt_5mc <- data.table::data.table(
    chrom  = character(),
    start0 = integer(),
    end    = integer(),
    score  = numeric()
  )
}
data.table::fwrite(bedgraph_dt_5mc, file = bedgraph_path_5mc, sep = "\t", col.names = FALSE)

bigwig_path_5mc <- file.path(region_dir, sprintf("%s.%s.5mC_fraction.bw", sample_name, region_slug))
if (nzchar(bedgraph_to_bigwig) && nrow(bedgraph_dt_5mc)) {
  conv_result_5mc <- system2(
    command = bedgraph_to_bigwig,
    args    = c(bedgraph_path_5mc, chrom_sizes_path, bigwig_path_5mc),
    stdout  = TRUE,
    stderr  = TRUE
  )
  conv_status_5mc <- attr(conv_result_5mc, "status")
  if (!is.null(conv_status_5mc) && conv_status_5mc != 0L) {
    warning(sprintf(
      "bedGraphToBigWig exited with status %d for %s.\nOutput:\n%s",
      conv_status_5mc, bigwig_path_5mc, paste(conv_result_5mc, collapse = "\n")
    ))
    if (file.exists(bigwig_path_5mc)) file.remove(bigwig_path_5mc)
    bigwig_path_5mc <- NULL
  }
} else {
  bigwig_path_5mc <- NULL
}

# Print the bigWig paths (or empty strings) so the calling pipeline can pick them up
cat(if (!is.null(bigwig_path)) bigwig_path else "", "\n", sep = "")
cat(if (!is.null(bigwig_path_5mc)) bigwig_path_5mc else "", "\n", sep = "")

# build region overlay tracks

potential problem: TAD are large, and no single fiber can span the whole TAD --> lose footprints level information if we plot the whole TAD
- If we want to retain the footprints level information, we can only eliminate TADs plot from .mcool file if we want to plot anything less than 500kb, and use the TAD boundary calls instead to plot the TAD boundaries as vertical lines on the plot

In [ ]:
import argparse
import csv
import gzip
import os
from collections import defaultdict


METADATA_DEFAULT = "/project/spott/1_Shared_projects/LCL_Fiber_seq/Data/LCL_sample_metatable_merged_samples_31samples.csv"
OUTPUT_ROOT_DEFAULT = "/project/spott/cshan/fiber-seq/results/TADs_m6a_footprint_overlay"
TAD_BED = "/project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pygenometracks_overlay/inputs/tads.bed"
CAGE_BED = "/project/spott/cshan/fiber-seq/results/TADs_m6a_footprint_overlay/inputs/cage_lcl_extended.bed.gz"
HIC_MATRIX = "/project/spott/cshan/annotations/hic/ENCFF216QQM.cool"
CTCF_BW = "/project/spott/cshan/annotations/CTCF_ENCFF636ODI.bigWig"
DNASE_BW = "/project/spott/cshan/annotations/DNase.ENCFF743ULW.bigWig"

FP_BINS = [
    ("fp_10_30bp", 10, 30, "#08306b", "Footprint 10-30bp"),
    ("fp_40_60bp", 40, 60, "#c6dbef", "Footprint 40-60bp"),
    ("fp_60_80bp", 60, 80, "#e377c2", "Footprint 60-80bp"),
    ("fp_140_160bp", 140, 160, "#ff7f0e", "Footprint 140-160bp"),
]


def normalize_chrom(chrom):
    chrom = str(chrom)
    return chrom if chrom.startswith("chr") else f"chr{chrom}"


def parse_region(region):
    chrom, coords = region.split(":", 1)
    start, end = coords.split("-", 1)
    return normalize_chrom(chrom), int(start), int(end)


def region_slug(chrom, start, end):
    return f"{chrom}_{start}_{end}"


def parse_args():
    parser = argparse.ArgumentParser(
        description="Build per-region per-read m6A and FiberHMM tracks plus a pyGenomeTracks config."
    )
    parser.add_argument("--sample", required=True)
    parser.add_argument("--region", default=None, help="chr:start-end")
    parser.add_argument("--chrom", default=None)
    parser.add_argument("--start", type=int, default=None)
    parser.add_argument("--end", type=int, default=None)
    parser.add_argument("--metadata", default=METADATA_DEFAULT)
    parser.add_argument("--output-root", default=OUTPUT_ROOT_DEFAULT)
    parser.add_argument("--m6a-summary-track", default=None, help="Optional region-level m6A bedGraph/bigWig from the R step.")
    parser.add_argument("--5mc-summary-track", dest="mc5_summary_track", default=None, help="Optional region-level 5mC bedGraph/bigWig from the R step.")
    parser.add_argument("--max-reads", type=int, default=250)
    parser.add_argument("--hic-depth", type=int, default=5000, help="Hi-C triangle depth in bp (default: 5000).")
    return parser.parse_args()


def load_sample_row(metadata_path, sample):
    with open(metadata_path, newline="") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            if row["sample_name"] == sample:
                return row
    raise SystemExit(f"Sample {sample} was not found in {metadata_path}")


def parse_int_list(value):
    return [int(x) for x in value.strip(",").split(",") if x]


def overlap(start, end, region_start, region_end):
    return max(0, min(end, region_end) - max(start, region_start))


def clip_interval(start, end, region_start, region_end):
    clipped_start = max(start, region_start)
    clipped_end = min(end, region_end)
    if clipped_end <= clipped_start:
        return None
    return clipped_start, clipped_end


def collect_m6a_records(path, chrom, region_start, region_end):
    records = {}
    with gzip.open(path, "rt") as handle:
        for line in handle:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 12 or fields[0] != chrom:
                continue
            row_start = int(fields[1])
            row_end = int(fields[2])
            if overlap(row_start, row_end, region_start, region_end) <= 0:
                continue
            block_sizes = parse_int_list(fields[10])
            block_starts = parse_int_list(fields[11])
            all_blocks = list(zip(block_starts, block_sizes))
            # First and last blocks are read-span anchors, not real m6A calls
            real_blocks = all_blocks[1:-1] if len(all_blocks) >= 2 else []
            records[fields[3]] = {
                "chrom": chrom,
                "name": fields[3],
                "strand": fields[5],
                "start": row_start,
                "end": row_end,
                "item_rgb": fields[8],
                "blocks": real_blocks,
            }
    return records


def collect_fp_records(path, chrom, region_start, region_end):
    records = defaultdict(list)
    with gzip.open(path, "rt") as handle:
        for line in handle:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 10 or fields[0] != chrom:
                continue
            row_start = int(fields[1])
            row_end = int(fields[2])
            if overlap(row_start, row_end, region_start, region_end) <= 0:
                continue
            read_id = fields[3]
            block_starts = parse_int_list(fields[8])
            block_sizes = parse_int_list(fields[9])
            records[read_id].append(
                {
                    "chrom": chrom,
                    "name": read_id,
                    "start": row_start,
                    "end": row_end,
                    "item_rgb": fields[6],
                    "blocks": list(zip(block_starts, block_sizes)),
                }
            )
    return records


def make_read_spans(m6a_records, fp_records, region_start, region_end):
    read_spans = {}
    for read_id, rec in m6a_records.items():
        clipped = clip_interval(rec["start"], rec["end"], region_start, region_end)
        if clipped is None:
            continue
        read_spans[read_id] = {
            "start": clipped[0],
            "end": clipped[1],
            "overlap_bp": clipped[1] - clipped[0],
            "feature_count": len(rec["blocks"]),
        }

    for read_id, recs in fp_records.items():
        starts = [rec["start"] for rec in recs]
        ends = [rec["end"] for rec in recs]
        clipped = clip_interval(min(starts), max(ends), region_start, region_end)
        if clipped is None:
            continue
        current = read_spans.setdefault(
            read_id,
            {
                "start": clipped[0],
                "end": clipped[1],
                "overlap_bp": clipped[1] - clipped[0],
                "feature_count": 0,
            },
        )
        current["start"] = min(current["start"], clipped[0])
        current["end"] = max(current["end"], clipped[1])
        current["overlap_bp"] = current["end"] - current["start"]
        current["feature_count"] += sum(len(rec["blocks"]) for rec in recs)

    return read_spans


def select_read_order(read_spans, max_reads):
    ordered = sorted(
        read_spans.items(),
        key=lambda item: (-item[1]["overlap_bp"], -item[1]["feature_count"], item[1]["start"], item[0]),
    )
    if max_reads and len(ordered) > max_reads:
        ordered = ordered[:max_reads]
    return [read_id for read_id, _ in ordered]


def aggregate_clipped_blocks(block_records, chrom_start, region_start, region_end, size_range=None, chrom_end=None):
    """
    Return blocks as (relative_start, size) pairs relative to chrom_start.

    chrom_start is the chromStart (col 2) of the BED12 row being written —
    i.e. span_start after clipping.  BED12 requires that the first block
    starts at 0 and the last block reaches chromEnd, so we insert 1-bp
    anchor blocks at those positions when no real block covers them.
    """
    clipped_blocks = []
    for record in block_records:
        for rel_start, size in record["blocks"]:
            if size_range is not None and not (size_range[0] <= size <= size_range[1]):
                continue
            block_start = record["start"] + rel_start
            block_end = block_start + size
            clipped = clip_interval(block_start, block_end, region_start, region_end)
            if clipped is None:
                continue
            relative = clipped[0] - chrom_start
            clipped_size = clipped[1] - clipped[0]
            clipped_blocks.append((relative, clipped_size))

    clipped_blocks.sort()

    # BED12 spec: first block relative start must be 0.
    if not clipped_blocks or clipped_blocks[0][0] != 0:
        clipped_blocks.insert(0, (0, 1))

    # BED12 spec: last block must reach chromEnd.
    if chrom_end is not None:
        required_end = chrom_end - chrom_start
        last_start, last_size = clipped_blocks[-1]
        if last_start + last_size < required_end:
            clipped_blocks.append((required_end - 1, 1))

    return clipped_blocks


def write_bed12(path, rows):
    with open(path, "w") as handle:
        for row in rows:
            handle.write("\t".join(map(str, row)) + "\n")


def build_tracks(sample, chrom, region_start, region_end, read_order, read_spans, m6a_records, fp_records, region_dir):
    baseline_rows = []
    m6a_rows = []
    fp_rows = {key: [] for key, *_ in FP_BINS}

    for read_id in read_order:
        span = read_spans[read_id]
        span_start = span["start"]
        span_end = span["end"]
        span_size = span_end - span_start
        if span_size <= 0:
            continue

        baseline_rows.append(
            [
                chrom,
                span_start,
                span_end,
                read_id,
                0,
                ".",
                span_start,
                span_end,
                "220,220,220",
                1,
                f"{span_size},",
                "0,",
            ]
        )

        if read_id in m6a_records:
            blocks = aggregate_clipped_blocks(
                [m6a_records[read_id]], span_start, region_start, region_end, chrom_end=span_end
            )
            if blocks:
                m6a_rows.append(
                    [
                        chrom,
                        span_start,
                        span_end,
                        read_id,
                        0,
                        m6a_records[read_id]["strand"],
                        span_start,
                        span_end,
                        "123,50,148",
                        len(blocks),
                        ",".join(str(size) for _, size in blocks) + ",",
                        ",".join(str(start) for start, _ in blocks) + ",",
                    ]
                )

        if read_id in fp_records:
            for key, lower, upper, _, _ in FP_BINS:
                blocks = aggregate_clipped_blocks(
                    fp_records[read_id], span_start, region_start, region_end, size_range=(lower, upper), chrom_end=span_end
                )
                # Only write a row if there are real footprint blocks beyond the anchor
                if len(blocks) > 1 or (len(blocks) == 1 and blocks[0][0] == 0 and blocks[0][1] > 1):
                    fp_rows[key].append(
                        [
                            chrom,
                            span_start,
                            span_end,
                            read_id,
                            0,
                            ".",
                            span_start,
                            span_end,
                            "0,0,0",
                            len(blocks),
                            ",".join(str(size) for _, size in blocks) + ",",
                            ",".join(str(start) for start, _ in blocks) + ",",
                        ]
                    )

    paths = {
        "baseline": os.path.join(region_dir, f"{sample}.{region_slug(chrom, region_start, region_end)}.reads_baseline.bed12"),
        "m6a": os.path.join(region_dir, f"{sample}.{region_slug(chrom, region_start, region_end)}.m6a_reads.bed12"),
    }
    write_bed12(paths["baseline"], baseline_rows)
    write_bed12(paths["m6a"], m6a_rows)

    for key in fp_rows:
        paths[key] = os.path.join(region_dir, f"{sample}.{region_slug(chrom, region_start, region_end)}.{key}.bed12")
        write_bed12(paths[key], fp_rows[key])

    return paths


def auto_cpg_bed(sample):
    path = f"/project/spott/cshan/fiber-seq/results/fire_CpG/{sample}/{sample}_CPG.combined.bed.gz"
    return path if os.path.exists(path) else None


def summary_track_type(path):
    lower = path.lower()
    if lower.endswith(".bw") or lower.endswith(".bigwig"):
        return "bigwig"
    if lower.endswith(".bedgraph") or lower.endswith(".bdg"):
        return "bedgraph"
    raise SystemExit(f"Unsupported m6A summary track type for {path}")


def write_tracks_ini(path, sample, chrom, track_paths, m6a_summary_track=None, mc5_summary_track=None, cpg_bed=None, hic_depth=200000):
    bigwig_root = (
        f"/project/spott/cshan/fiber-seq/results/PolII/FIRE_combined_footprints/"
        f"joint_trained_tracks/{chrom}"
    )
    with open(path, "w") as handle:
        handle.write("[x-axis]\nfontsize = 10\nwhere = top\n\n")

        handle.write("[hic]\n")
        handle.write(f"file = {HIC_MATRIX}\n")
        handle.write("title = GM12878 Hi-C (5 kb)\n")
        handle.write(f"depth = {hic_depth}\nheight = 6\ncolormap = Reds\nfile_type = hic_matrix\n\n")

        handle.write("[spacer]\nheight = 0.3\n\n")

        handle.write("[tads]\n")
        handle.write(f"file = {TAD_BED}\n")
        handle.write("title = TADs (Arrowhead, GM12878)\n")
        handle.write("height = 2\ncolor = none\nborder_color = #e41a1c\nline_width = 1.5\nlabels = false\ndisplay = interleaved\nfile_type = bed\n\n")

        for title, file_path, color in [
            ("CTCF", CTCF_BW, "black"),
            ("DNase", DNASE_BW, "#1f77b4"),
        ]:
            handle.write("[spacer]\nheight = 0.3\n\n")
            handle.write(f"[{title.lower()}]\n")
            handle.write(f"file = {file_path}\n")
            handle.write(f"title = {title}\n")
            handle.write(f"height = 2\ncolor = {color}\nmin_value = 0\nshow_data_range = true\nfile_type = bigwig\n\n")

        handle.write("[spacer]\nheight = 0.3\n\n")
        handle.write("[cage]\n")
        handle.write(f"file = {CAGE_BED}\n")
        handle.write("title = FANTOM5 CAGE\n")
        handle.write("height = 1.5\ncolor = #2ca02c\nborder_color = #1b6e1b\nlabels = false\ndisplay = collapsed\nfile_type = bed\n\n")

        if cpg_bed is not None:
            handle.write("[spacer]\nheight = 0.3\n\n")
            handle.write("[cpg]\n")
            handle.write(f"file = {cpg_bed}\n")
            handle.write(f"title = CpG methylation pb-CpG ({sample})\n")
            handle.write("height = 1.5\ncolor = #8856a7\nmin_value = 0\nmax_value = 100\nshow_data_range = true\nfile_type = bedgraph\n\n")

        for key, _, _, color, title in FP_BINS:
            handle.write("[spacer]\nheight = 0.3\n\n")
            handle.write(f"[{key}_bw]\n")
            handle.write(f"file = {bigwig_root}/combined_{chrom}_{title.split()[-1]}_fp_cov.bw\n")
            handle.write(f"title = {title}\n")
            handle.write(f"height = 1.2\ncolor = {color}\nmin_value = 0\nshow_data_range = true\nfile_type = bigwig\n\n")

        if (
            m6a_summary_track is not None
            and os.path.exists(m6a_summary_track)
            and os.path.getsize(m6a_summary_track) > 0
        ):
            handle.write("[spacer]\nheight = 0.3\n\n")
            handle.write("[m6a_summary]\n")
            handle.write(f"file = {m6a_summary_track}\n")
            handle.write("title = m6A fraction (footprintR)\n")
            handle.write("height = 1.6\ncolor = #7B3294\nmin_value = 0\nmax_value = 1\nshow_data_range = true\n")
            handle.write(f"file_type = {summary_track_type(m6a_summary_track)}\n\n")

        if (
            mc5_summary_track is not None
            and os.path.exists(mc5_summary_track)
            and os.path.getsize(mc5_summary_track) > 0
        ):
            handle.write("[spacer]\nheight = 0.3\n\n")
            handle.write("[5mc_summary]\n")
            handle.write(f"file = {mc5_summary_track}\n")
            handle.write("title = 5mC fraction (footprintR)\n")
            handle.write("height = 1.6\ncolor = #2c7bb6\nmin_value = 0\nmax_value = 1\nshow_data_range = true\n")
            handle.write(f"file_type = {summary_track_type(mc5_summary_track)}\n\n")

        handle.write("[spacer]\nheight = 0.3\n\n")
        handle.write("[reads_baseline]\n")
        handle.write(f"file = {track_paths['baseline']}\n")
        handle.write("title = Per-read overlay\n")
        handle.write("height = 14\ncolor = #d9d9d9\nborder_color = #d9d9d9\nlabels = false\ndisplay = stacked\nfile_type = bed\n\n")

        handle.write("[m6a_reads]\n")
        handle.write(f"file = {track_paths['m6a']}\n")
        handle.write("title = \n")
        handle.write("height = 14\ncolor = #7B3294\nborder_color = #7B3294\nlabels = false\ndisplay = stacked\noverlay_previous = share-y\nfile_type = bed\n\n")

        for key, _, _, color, title in FP_BINS:
            handle.write(f"[{key}]\n")
            handle.write(f"file = {track_paths[key]}\n")
            handle.write("title = \n")
            handle.write(f"height = 14\ncolor = {color}\nborder_color = {color}\nlabels = false\ndisplay = stacked\noverlay_previous = share-y\nfile_type = bed\n\n")


def main():
    args = parse_args()
    if args.region:
        chrom, region_start, region_end = parse_region(args.region)
    else:
        if args.chrom is None or args.start is None or args.end is None:
            raise SystemExit("Provide --region or --chrom/--start/--end")
        chrom = normalize_chrom(args.chrom)
        region_start = args.start
        region_end = args.end

    if region_start < 1 or region_end < region_start:
        raise SystemExit("Invalid genomic coordinates")

    sample_row = load_sample_row(args.metadata, args.sample)
    fire_dir = sample_row["fire_dir"]
    region_name = region_slug(chrom, region_start, region_end)
    region_dir = os.path.join(args.output_root, args.sample, chrom, region_name)
    os.makedirs(region_dir, exist_ok=True)

    m6a_path = os.path.join(
        fire_dir, "extracted_results", "m6a_by_chr", f"{args.sample}.ft_extracted_m6a.{chrom}.bed.gz"
    )
    fp_path = os.path.join(
        "/project/spott/1_Shared_projects/LCL_Fiber_seq/FiberHMM/merged",
        args.sample,
        "joint_trained_results",
        f"{args.sample}_{chrom}_fp.bed.gz",
    )

    if not os.path.exists(m6a_path):
        raise SystemExit(f"Missing sample m6A BED12: {m6a_path}")
    if not os.path.exists(fp_path):
        raise SystemExit(f"Missing sample FiberHMM footprint BED: {fp_path}")

    m6a_records = collect_m6a_records(m6a_path, chrom, region_start, region_end)
    fp_records = collect_fp_records(fp_path, chrom, region_start, region_end)
    read_spans = make_read_spans(m6a_records, fp_records, region_start, region_end)
    read_order = select_read_order(read_spans, args.max_reads)
    if not read_order:
        raise SystemExit("No reads overlapped the requested region.")

    track_paths = build_tracks(
        args.sample, chrom, region_start, region_end, read_order, read_spans, m6a_records, fp_records, region_dir
    )

    read_table_path = os.path.join(region_dir, f"{args.sample}.{region_name}.selected_reads.tsv")
    with open(read_table_path, "w", newline="") as handle:
        writer = csv.writer(handle, delimiter="\t")
        writer.writerow(["read_id", "clipped_start", "clipped_end", "overlap_bp", "feature_count"])
        for read_id in read_order:
            row = read_spans[read_id]
            writer.writerow([read_id, row["start"], row["end"], row["overlap_bp"], row["feature_count"]])

    ini_path = os.path.join(region_dir, f"{args.sample}.{region_name}.tracks.ini")
    write_tracks_ini(
        ini_path,
        sample=args.sample,
        chrom=chrom,
        track_paths=track_paths,
        m6a_summary_track=args.m6a_summary_track,
        mc5_summary_track=args.mc5_summary_track,
        cpg_bed=auto_cpg_bed(args.sample),
        hic_depth=args.hic_depth,
    )

    print(ini_path)


if __name__ == "__main__":
    main()

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
import pysam
import pyBigWig
from collections import defaultdict
import os

In [17]:
# inspect promoter range

cage_expanded_tss_df = pd.read_csv("/project/spott/cshan/fiber-seq/results/TADs_m6a_footprint_overlay/inputs/cage_lcl_extended.bed.gz", sep="\t", header=None)

# keep the first 3 columns and rename them to chrom, start, end
cage_expanded_tss_df = cage_expanded_tss_df.iloc[:, :3]
cage_expanded_tss_df.columns = ["cage_chrom", "cage_start", "cage_end"]
cage_expanded_tss_df.head()

,cage_chrom,cage_start,cage_end
0,chr1,633908,634108
1,chr1,633909,634109
2,chr1,633926,634126
3,chr1,633927,634127
4,chr1,633928,634128


In [18]:
# inspect tads range
tads_df = pd.read_csv("/project/spott/cshan/fiber-seq/results/TADs_Fiber_MSP/pygenometracks_overlay/inputs/tads.bed", sep="\t", header=None)

tads_df = tads_df.iloc[:, :3]
tads_df.columns = ["tad_chrom", "tad_start", "tad_end"]
tads_df.head()

,tad_chrom,tad_start,tad_end
0,chr1,915000,1005000
1,chr1,1030000,1235000
2,chr1,1255000,1450000
3,chr1,1710000,1840000
4,chr1,1860000,2055000


In [23]:
# find the TAD that overlaps the promoter region

import pyranges as pr

cage_pr = pr.PyRanges(cage_expanded_tss_df.rename(columns={
    "cage_chrom": "Chromosome",
    "cage_start": "Start",
    "cage_end": "End"
}))

tad_pr = pr.PyRanges(tads_df.rename(columns={
    "tad_chrom": "Chromosome",
    "tad_start": "Start",
    "tad_end": "End"
}))

overlap = cage_pr.join(tad_pr)

# rename columns back to original names
overlap_df = overlap.df.rename(columns={
    "Chromosome": "chrom",
    "Start": "cage_start",
    "End": "cage_end",
    "Start_b": "tad_start",
    "End_b": "tad_end"
})  

# filter for unique TADs
unique_tads = overlap_df.drop_duplicates(
    subset=["chrom", "tad_start", "tad_end"]
)
unique_tads.head()

,chrom,cage_start,cage_end,tad_start,tad_end
0,chr1,959150,959350,915000,1005000
17,chr1,1033250,1033450,1030000,1235000
37,chr1,1273726,1273926,1255000,1450000
144,chr1,1724226,1724426,1710000,1840000
158,chr1,1890942,1891142,1860000,2055000


In [6]:
%%bash

sbatch /project/spott/cshan/fiber-seq/code/TADs_m6a_footprint_overlay_code/run_region_overlay.sh \
  --sample AL60_bc2152_18519 \
  --region chr1:1290000-1320000 

sbatch: Verify job submission ...
sbatch: Using a shared partition ...
sbatch: Partition: caslake
sbatch: QOS-Flag: caslake
sbatch: Account: pi-spott
sbatch: Verification: ***PASSED***


Submitted batch job 49057871
